# <center>**Chapter 18 - Datasets and DataLoaders in PyTorch**<center>

Notebook làm theo ví dụ Sonar trong chương: nạp dữ liệu, tạo Dataset, chia tập bằng random_split, rồi dùng DataLoader để tạo các batch huấn luyện.

In [ ]:
import csv
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

def find_data_file(filename):
    start = Path.cwd().resolve()
    patterns = (f"data/**/{filename}", f"*/data/**/{filename}", f"*/*/data/**/{filename}")
    for root in (start, *start.parents):
        for pattern in patterns:
            matches = list(root.glob(pattern))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Cannot find {filename} near the current directory")

In [ ]:
with find_data_file("sonar.all-data").open(newline="", encoding="utf-8-sig") as file:
    rows = list(csv.reader(file))
features = [[float(value) for value in row[:60]] for row in rows]
# Mã hóa nhãn nhị phân: M=0, R=1.
targets = [[1.0 if row[60] == "R" else 0.0] for row in rows]

In [ ]:
class SonarDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [ ]:
torch.manual_seed(42)
dataset = SonarDataset(features, targets)
train_set, test_set = random_split(
    dataset, [0.7, 0.3], generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_set, shuffle=True, batch_size=16)
test_loader = DataLoader(test_set, shuffle=False, batch_size=16)
print(f"Samples: {len(dataset)} | train: {len(train_set)} | test: {len(test_set)}")
print("Dataset sample shapes:", dataset[0][0].shape, dataset[0][1].shape)
print("Batch shapes:", *(tensor.shape for tensor in next(iter(train_loader))))

Samples: 208 | train: 146 | test: 62
Dataset sample shapes: torch.Size([60]) torch.Size([1])
Batch shapes: torch.Size([16, 60]) torch.Size([16, 1])


In [ ]:
model = nn.Sequential(
    nn.Linear(60, 60),
    nn.ReLU(),
    nn.Linear(60, 30),
    nn.ReLU(),
    nn.Linear(30, 1),
    nn.Sigmoid(),
)
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
epochs = 200

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    sample_count = 0
    for X_batch, y_batch in train_loader:
        prediction = model(X_batch)
        loss = loss_fn(prediction, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        sample_count += len(y_batch)
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d}/{epochs} | train BCE={total_loss / sample_count:.4f}")

Epoch  50/200 | train BCE=0.4770
Epoch 100/200 | train BCE=0.3949
Epoch 150/200 | train BCE=0.2635
Epoch 200/200 | train BCE=0.1565


In [ ]:
model.eval()
correct = 0
sample_count = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predicted_labels = (model(X_batch) >= 0.5).float()
        correct += (predicted_labels == y_batch).sum().item()
        sample_count += len(y_batch)
print(f"Test accuracy: {correct / sample_count:.2%} ({correct}/{sample_count})")

Test accuracy: 80.65% (50/62)


In [12]:
# Dự đoán trên một batch test: xác suất > 0.5 tương ứng lớp Rock (R).
model.eval()
with torch.no_grad():
    for features, actual in list(test_loader)[:1]:
        probabilities = model(features).flatten()
        for probability, label in zip(probabilities[:5], actual[:5].flatten()):
            predicted = "R" if probability.item() >= 0.5 else "M"
            expected = "R" if label.item() >= 0.5 else "M"
            print(f"predicted={predicted}, actual={expected}, p(R)={probability.item():.3f}")

predicted=R, actual=R, p(R)=0.998
predicted=R, actual=R, p(R)=0.999
predicted=R, actual=M, p(R)=0.881
predicted=M, actual=R, p(R)=0.337
predicted=R, actual=R, p(R)=1.000


Theo chương: nạp Sonar, mã hóa nhãn M/R, hiện thực Dataset với `__len__` và `__getitem__`, chia train/test bằng `random_split`, rồi huấn luyện MLP nhị phân bằng DataLoader có shuffle và batch size 16.

# <center>**HẾT**<center>